# Tutorial 3g: Case Study — Cooke Triplet


### June 2025


This end-to-end case study walks through a complete lens design workflow using the modern
Optiland optimization API:

1. Build a Cooke triplet starting design
2. Define operands (RMS spot size, focal length)
3. Define variables (radii, thicknesses, image distance)
4. Run `minimize()` with DLS
5. Inspect the `OptimizationResult`
6. Compare before/after with `lens.draw()` and `SpotDiagram`

A **Cooke triplet** is the simplest three-element design capable of correcting the five
primary Seidel aberrations simultaneously.  It consists of a positive crown element, a
negative flint element near the stop, and a second positive crown element.

This tutorial assumes familiarity with Tutorial 3a (basic optimization) and Tutorial 3d
(result inspection).  If you are new to Optiland optimization, start there.


In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt

from optiland import optic, analysis
from optiland.optimization import minimize, OptimizationProblem
from optiland.optimization.observers.history import HistoryObserver
from optiland.optimization.stopping.criteria import MaxIter, CostTolerance


## 1. Build the starting design

We begin with a classical Cooke triplet geometry.  The initial radii are rough estimates;
optimization will refine them.  Glass types are fixed at N-SK16 (positive elements) and
N-SF5 (negative element) — standard Cooke choices.

System specification:

- Focal length: 100 mm
- F-number: ~8.3 (EPD = 12 mm)
- Full field of view: ±1° (angle field)
- Three design wavelengths: 486 nm, 588 nm, 656 nm


In [ ]:
lens = optic.Optic()

# Object at infinity
lens.surfaces.add(index=0, thickness=np.inf)

# Element 1 — positive crown (N-SK16)
lens.surfaces.add(index=1, thickness=6.0, radius=25.0, material='N-SK16')
lens.surfaces.add(index=2, thickness=0.5, radius=-100.0)

# Element 2 — negative flint (N-SF5), stop on front surface
lens.surfaces.add(index=3, thickness=2.5, radius=-20.0, material='N-SF5')
lens.surfaces.add(index=4, thickness=2.0, radius=20.0, is_stop=True)

# Element 3 — positive crown (N-SK16)
lens.surfaces.add(index=5, thickness=2.5, radius=100.0, material='N-SK16')
lens.surfaces.add(index=6, thickness=0.5, radius=-25.0)

# Image distance
lens.surfaces.add(index=7, thickness=55.0)

# Image plane
lens.surfaces.add(index=8)

# System definition
lens.set_aperture(aperture_type='EPD', value=12)
lens.fields.set_type('angle')
lens.fields.add(y=0.0)
lens.fields.add(y=0.7)
lens.fields.add(y=1.0)
lens.wavelengths.add(value=0.4861)
lens.wavelengths.add(value=0.5876, is_primary=True)
lens.wavelengths.add(value=0.6563)

lens.update_paraxial()

print('Starting EFL:', lens.paraxial.EFL)


## 2. Draw the starting design


In [ ]:
# Snapshot the starting state before any optimization
starting = copy.deepcopy(lens)

print('=== Starting design ===')
lens.draw()


## 3. Define operands

We optimise for:

- **RMS spot size** at each of the three normalised fields (0, 0.7, 1.0) at the primary
  wavelength — this is the main image quality metric.
- **Effective focal length** = 100 mm — maintains the system power during optimisation.


In [ ]:
problem = OptimizationProblem()

# RMS spot size at each field
for Hx, Hy in lens.fields.get_field_coords():
    input_data = {
        'optic': lens,
        'surface_number': -1,
        'Hx': Hx,
        'Hy': Hy,
        'num_rays': 5,
        'wavelength': 0.5876,
        'distribution': 'hexapolar',
    }
    problem.add_operand('rms_spot_size', target=0, weight=1, input_data=input_data)

# Focal length constraint
problem.add_operand(
    'effective_focal_length',
    target=100,
    weight=1,
    input_data={'optic': lens},
)

print(f'Number of operands: {len(problem.operands)}')
print(f'Starting merit: {problem.merit_function_value:.4f}')


## 4. Define variables

We free all six surface radii and the air spaces between and after the elements, plus the
image distance.  Element centre thicknesses (surfaces 1, 3, 5) are kept fixed to maintain
manufacturability constraints.


In [ ]:
# All six refractive surface radii
for s in [1, 2, 3, 4, 5, 6]:
    problem.add_variable(lens, 'radius', surface_number=s, min_val=-500, max_val=500)

# Air spaces (surfaces 2, 4, 6)
for s in [2, 4, 6]:
    problem.add_variable(lens, 'thickness', surface_number=s, min_val=0.1, max_val=20.0)

# Image distance
problem.add_variable(lens, 'thickness', surface_number=7, min_val=40.0, max_val=120.0)

print(f'Number of variables: {len(problem.variables)}')
problem.info()


## 5. Optimize with DLS

Damped Least-Squares (DLS) is the industry-standard algorithm for optical design.  It
combines a Jacobian-based least-squares step with a damping parameter that blends between
gradient descent (stable far from the minimum) and Gauss-Newton (fast near the minimum).

We attach a `HistoryObserver` so we can plot convergence after the run.


In [ ]:
history_obs = HistoryObserver()
stop = MaxIter(150) | CostTolerance(1e-7)

result = minimize(
    problem,
    'dls',
    observers=[history_obs],
    stop=stop,
)

print(result)


## 6. Inspect the result


In [ ]:
print(f'Method:         {result.method}')
print(f'Stop reason:    {result.stop_reason}')
print(f'Wall time:      {result.wall_time_s:.2f}s')
print()
print(f'Starting merit: {history_obs.history[0]:.4f}')
print(f'Final merit:    {result.value:.4f}')
print(f'Improvement:    {result.improvement_pct:.1f}%')
print(f'Success:        {result.success}')


## 7. Convergence plot


In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(history_obs.history, color='steelblue', linewidth=1.5)
plt.xlabel('Iteration')
plt.ylabel('Merit (log scale)')
plt.title('DLS convergence — Cooke triplet')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 8. Draw the optimised design


In [ ]:
print('=== Optimised design ===')
lens.draw()


## 9. Spot diagrams — before and after

A spot diagram shows where rays from each field strike the image plane.  Smaller, tighter
clusters indicate better image quality.  We compare the starting and optimised designs
side-by-side.


In [ ]:
print('=== Spot diagram — STARTING design ===')
spot_start = analysis.SpotDiagram(starting)
spot_start.view()


In [ ]:
print('=== Spot diagram — OPTIMISED design ===')
spot_opt = analysis.SpotDiagram(lens)
spot_opt.view()


In [ ]:
# Print RMS spot radii for a quantitative comparison
fields = lens.fields.get_field_coords()
wavelengths = lens.wavelengths.get_wavelengths()

rms_start = spot_start.rms_spot_radius()
rms_opt = spot_opt.rms_spot_radius()

print('RMS spot radius comparison (mm):')
print(f'{"Field":>12}  {"Wavelength":>12}  {"Starting":>12}  {"Optimised":>12}  {"Improvement":>12}')
for i, field in enumerate(fields):
    for j, wave in enumerate(wavelengths):
        r0 = rms_start[i][j]
        r1 = rms_opt[i][j]
        pct = 100 * (r0 - r1) / r0 if r0 > 0 else float('nan')
        print(f'{str(field):>12}  {wave:>12.4f}  {r0:>12.5f}  {r1:>12.5f}  {pct:>11.1f}%')


## Conclusions

- Starting from a rough Cooke triplet geometry we drove the RMS spot size down
  substantially using the DLS algorithm with a two-criterion stopping rule.
- The `OptimizationResult` provided transparent diagnostics: stop reason, wall time,
  improvement percentage, and the final parameter vector.
- `HistoryObserver` let us plot the full convergence curve with zero extra boilerplate.
- The workflow — build lens, define operands, define variables, call `minimize()`, inspect
  result — is the same for any system, from a simple singlet to a complex zoom lens.

Next steps to explore:

- Add more fields and wavelengths to the merit function for better colour correction.
- Follow up with `GlassExpert` (Tutorial 3f) to optimise glass selection.
- Use a global solver (`differential_evolution`) as a first stage to escape local minima,
  then refine with DLS.
- Run tolerancing analysis to assess manufacturing sensitivity.
